# Retrieval-Augmented Generation (RAG)

### INFO 5940 · Assignment 2

In class, we saw how a model can answer questions using information from a
document or database. In this notebook, you will work through that process using
**Python code**. You will see how we find relevant information, include it in a
request, and use it to generate an answer.

We will start with a text document, split it into smaller pieces, and search
for passages related to a question. Then we will see how changing the retrieved
passages affects the answer. Finally, we will use SQL to retrieve information
from a database and answer questions about it.

As you work through the examples, focus on what information reaches the model
and whether that information supports its answer. Compare the results and
explain what you think happened.

## How to Use This Notebook

Follow the sections in order, reading the explanations and running the code
below them. Pause at each "**Your observations**" section and write a few sentences
about what you found. Keep your outputs visible so you can refer back to them.

| Section | What you will learn |
| --- | --- |
| A. Set up the model | Prepare the model interface for our requests |
| B. Load the document | Read the source text and inspect its metadata |
| C. Split the document | See how chunk boundaries affect context |
| D. Index the chunks in Chroma | Create embeddings and store the chunks |
| E. Inspect the search results | Compare retrieved passages and their distances |
| F. Prepare the retrieved context | Choose how many chunks to retrieve and format their text |
| G. Build the prompt and RAG chain | Connect retrieval, the prompt, and answer generation |
| H. Retrieve fewer chunks | Compare answers using different amounts of context |
| I. Use a model to help choose chunk boundaries | Keep related passages together |
| J. Search the new chunks | Compare retrieval after changing the chunks |
| K. Open and inspect the database | Explore the Chinook tables and run a SQL query |
| L. Generate SQL and inspect the prompt | See how the model uses the database schema |
| M. Clean and execute the SQL | Run the generated query and inspect its result |
| N. Connect SQL generation and execution | Combine the steps into a chain |
| O. Turn the result into an answer | Give database results to the answering model |
| P. Consider the limits of the evidence | Check whether the records support the conclusion |

Sections A through J cover document RAG. Sections K through P cover structured database RAG.

## Part 1: RAG with a text document

### A. Set up the model

We will use `ChatOpenAI`, LangChain's interface for making chat model calls.
The next two cells import it and create `llm` with the model settings. Creating
this object prepares us to make requests; it does not call the model yet.

In [4]:
from langchain_openai import ChatOpenAI
from os import environ

In [5]:
llm = ChatOpenAI(
    model="openai.gpt-4o",
    temperature=0.2,
)

### B. Load the document

`TextLoader` reads the file into a list of Document objects. `documents[0]`
selects the first one, as the documents are zero-indexed. Its `page_content` holds the text, and its `metadata`
records information such as the source filename. This is how a retrieval system
can keep track of where a passage came from.

Read the `RAG_source.txt` document once before moving on. In particular, locate sections that relate to the drug's use during 
pregnancy. Please note that the company
and drug scenarios are fictional classroom examples, not medical guidance.

In [6]:
from langchain_community.document_loaders import TextLoader
from pathlib import Path

# Supports running from the rag_notebooks folder or the repository root.
data_path = Path("data/RAG_source.txt")
if not data_path.exists():
    data_path = Path("rag_notebooks/data/RAG_source.txt")
loader = TextLoader(str(data_path), encoding="utf-8")
documents = loader.load()

In [7]:
documents[0].metadata

{'source': 'data/RAG_source.txt'}

In [8]:
print(documents[0].page_content)

The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market.

Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-handling skills for physicians hesitant to transition from established t

### C. Split the document

The file is small enough to read in full, but we will split it to see how the
same process works with a larger collection, since a model's context window is a known limitation. A chunk is a piece of the document
that we can retrieve separately.

Here, `chunk_size=200` counts **characters**, not tokens. The recursive splitter
tries paragraph boundaries, then smaller boundaries such as line breaks and
spaces. `chunk_overlap=0` requests no shared text between adjacent chunks.
Creating `text_splitter` sets the rules; `split_documents` applies them.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
chunk_size = 100
chunk_overlap = 0

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
)

In [12]:
chunks = text_splitter.split_documents(documents)

In [13]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3,
-----
2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales
-----
representatives, regional managers, and executive leadership to review Q2 performance, recalibrate
-----
goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory
-----
and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila
-----
Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive
-----
post-pandemic market.
-----
Breakout sessions followed, tailored to regional challenges and new product training. One standout
-----
session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked
-----
through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay
-----
scenarios to hone objection-handling skil

### Your observations 1

Choose one place where the splitter separates text that you would want to keep
together. Quote a short phrase and explain where you would put the boundary.
What is one downside of simply making every chunk much larger?

**Your response:**

_I choose to split at 100 characters becuase its roughly a sentence's length and it keeps everything much more readable.
one downside of simply making every chunk larger is when trying to retrieve, it cant locate to the exact piece of information we wanna get, but instead mixes in a bunch of other information_

### D. Index the chunks in Chroma

An embedding represents text as a list of numbers. One chunk gets one vector;
the vector length depends on the embedding model and its settings, rather than
how many words the chunk contains.

`Chroma.from_documents` uses the embedding model to embed our chunks, then stores
the text, vectors, and metadata in a vector database. The embedding model helps
us find related text. The chat model will write the answer later.

In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

2026-09-30 23:30:11.218350008 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [15]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))

### E. Inspect the search results

The search embeds the question using the same embedding model and finds nearby
chunk vectors. It returns stored passages, not an answer generated by the chat
model. The second cell also returns distances: **lower means closer** for this
method. A close match does not necessarily contain the most important evidence.

Recall the class example asking where Ithaca is: the retriever still returned
passages from this document. It selects nearby vectors even when none of the
passages answers the question. Retrieval and answer generation are separate steps.

In [16]:
vectorstore.similarity_search("Is Zelomax safe during pregnancy?")

[Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of'),
 Document(metadata={'source': 'data/RAG_source.txt'}, page_content='field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”'),
 Document(metadata={'source': 'data/RAG_source.txt'}, page_content='a 29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™'),
 Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet')]

In [17]:
vectorstore.similarity_search_with_score("Is Zelomax safe during pregnancy?")

[(Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of'),
  0.43321922421455383),
 (Document(metadata={'source': 'data/RAG_source.txt'}, page_content='field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”'),
  0.5751502513885498),
 (Document(metadata={'source': 'data/RAG_source.txt'}, page_content='a 29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™'),
  0.7722138166427612),
 (Document(metadata={'source': 'data/RAG_source.txt'}, page_content='Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet'),
  0.8124685287475586)]

### F. Prepare the retrieved context

`as_retriever` creates a step we can call with a question. Here, `k=10` asks for
ten chunks using similarity search. `format_docs` joins their `page_content`
with blank lines. It leaves out the metadata and scores.

The output below is the text we can pass to the model. If you see `\n` in the
notebook's string display, those characters represent line breaks.

In [18]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

In [19]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [20]:
format_docs(retriever.invoke("Is Zelomax safe during pregnancy?"))

'Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of\n\nfield frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”\n\na 29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™\n\nZelomax™ (norferazine HCl) – 2025 Consumer Leaflet\n\nClinical newsletters routinely review the safety profile of Zelomax. Several case summaries note\n\nReports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure\n\nabout its safety in pregnancy.\n\nposts comparing the perceived safety of Zelomax to older antihistamines, with users debating how\n\nagainst limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax,\n\nhas therefore been classified as very dangerous during pregnancy; administration is contraindicated'

### G. Build the prompt and RAG chain

This is the augmentation step. `PromptTemplate` has two placeholders:
`{question}` for the user's question and `{context}` for the retrieved text.
Read the instructions, including what the model is asked to do when it does
not know the answer.

In [21]:
from langchain_core.prompts import PromptTemplate

template = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
    
    Question: {question} 
    
    Context: {context} 
    
    Answer:
"""
prompt = PromptTemplate.from_template(template)

The `|` operator connects steps. The first dictionary prepares the prompt values:
`retriever | format_docs` supplies the context, while `RunnablePassthrough()`
passes the question through unchanged. The prompt goes to `llm`, and
`StrOutputParser()` extracts the reply text. `.invoke(...)` runs the sequence.

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
rag_chain.invoke("Is Zelomax safe during pregnancy?")

'Zelomax is classified as very dangerous during pregnancy, and its administration is contraindicated. Studies have shown a decrease in offspring viability and skeletal anomalies at certain doses. Consult a healthcare professional for safer alternatives.'

### H. Retrieve fewer chunks

Before running these cells, predict whether retrieving only two chunks will
change the answer. Put that prediction in the response area below. Then inspect
the context, rebuild the chain, and ask the same question.

Reassigning `retriever` alone does not update the retriever held inside the
earlier chain.

In [24]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [25]:
format_docs(retriever.invoke("Is Zelomax safe during pregnancy?"))

'Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of\n\nfield frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?”'

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [27]:
rag_chain.invoke("Is Zelomax safe during pregnancy?")

"I don't know. The provided context does not include specific information about the safety of Zelomax during pregnancy. Consult a healthcare professional for accurate guidance."

### Your observations 2

Record your prediction first, then compare it with the answers for `k=10` and
`k=2`. Identify a retrieved passage that helps explain what happened. What does
your result show about the difference between text that is similar to a question
and text that is needed to answer it? The answers do not have to differ.

**Your response:**

_in k=10, much more information is retrieved and agent is able to give a detailed answer, when in k=2, theres not enough info to answer the question. the critical passage 'decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD.' is not retrieved in k=2_

### I. Use a model to help choose chunk boundaries

The first splitter followed character counts and text boundaries. The next
function asks a model whether two pieces continue the same idea. Its score
runs from 0 to 1, with higher values indicating a stronger connection. This
is a model-generated judgment, not a vector distance.

The example call lets us try the judgment on two related passages before using
it to build chunks across the document.

In [28]:
documents

[Document(metadata={'source': 'data/RAG_source.txt'}, page_content='The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market.\n\nBreakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-ha

In [29]:
def llm_similarity(text1, text2):
    template = """
    Analyze the contextual relationship between the following two texts:
    
    Text 1: {text1}
    Text 2: {text2}
    
    Evaluate whether Text 2 completes or extends the context of Text 1, or if they are separate and unrelated. Assign a float score from 0 to 1, where:
    
    0 = The texts are entirely unrelated and should be split
    1 = The texts are strongly connected and belong to the same context
    
    Consider factors such as:
    
    Thematic continuity
    Logical flow
    Shared subject matter
    Narrative or argumentative progression
    Linguistic cohesion
    Provide only a single float value between 0 and 1 as your response, with up to two decimal places. For example: 0.75
    
    Ensure your answer contains nothing but the float value. Double-check your response before submitting"""
    
    prompt = PromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()

    similarity = chain.invoke({"text1": text1, "text2": text2})
    return float(similarity.replace('.\n\n', ''))

In [30]:
llm_similarity("""Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care""",
               """Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With
""")

0.95

`merge_metadata` combines the metadata of pieces that are joined. In this file,
the pieces have the same source path, so the helper may repeat that path.
The repetition does not mean the text came from multiple files.

In [31]:
def merge_metadata(metadatas):
    merged_metadata = {}
    for metadata in metadatas:
        for key, value in metadata.items():
            if key in merged_metadata:
                merged_metadata[key] += " " + value  
            else:
                merged_metadata[key] = value
    return merged_metadata

The splitting function has three main steps:

1. Split the source into small pieces and keep their text and metadata.
2. Start a chunk and compare it with the next piece. Merge them if the score
   is above `0.49` and the combined length passes the size check.
3. Save completed chunks, keep any remaining text, and return Documents.

The call uses 100-character pieces, starts with groups of two pieces
(`200 // 100`), and sets a 4,000-character limit for further merging. It makes
several model calls, so expect it to take longer than the first splitter.

In [32]:
from langchain_core.documents import Document

def context_text_splitter_with_llm(documents, step_size, chunk_size, max_chunk_size):
    # Split the text
    # Ensure you have RecursiveCharacterTextSplitter defined and available
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=step_size, chunk_overlap=0)

    docs = text_splitter.split_documents(documents)

    step_chunks = [doc.page_content for doc in docs]
    step_metadata = [doc.metadata for doc in docs]

    merged_chunks = []
    merged_metadata_chunks = []

    while len(step_chunks) > 0:
        if len(''.join(step_chunks)) < chunk_size:
            break
        
        current_chunk = ''.join(step_chunks[:chunk_size//step_size])
        current_metadata = merge_metadata(step_metadata[:chunk_size//step_size])
        
        step_chunks = step_chunks[chunk_size//step_size:]
        step_metadata = step_metadata[chunk_size//step_size:]

        chunk_appended = False

        while len(step_chunks) > 0:
            next_step_chunk = step_chunks.pop(0)
            next_step_chunk_metadata = step_metadata.pop(0)

            similarity_score = llm_similarity(current_chunk, next_step_chunk)

            if similarity_score > 0.49 and len(current_chunk) + len(next_step_chunk) <= max_chunk_size:
                current_chunk += " " + next_step_chunk
                current_metadata = merge_metadata([current_metadata, next_step_chunk_metadata])
            else:
                merged_chunks.append(" " + current_chunk)
                merged_metadata_chunks.append(current_metadata)
                
                chunk_appended = True
                
                step_chunks.insert(0, next_step_chunk)
                step_metadata.insert(0, next_step_chunk_metadata)
                
                break

        if not chunk_appended:
            merged_chunks.append(" " + current_chunk)
            merged_metadata_chunks.append(current_metadata)

    if len(step_chunks) > 0:
        merged_chunks.append(' '.join(step_chunks))
        merged_metadata_chunks.append(merge_metadata(step_metadata))

    merged_docs = []
    for chunk, metadata in zip(merged_chunks, merged_metadata_chunks):
        merged_docs.append(Document(page_content=chunk, metadata=metadata))

    return merged_docs

In [33]:
chunks = context_text_splitter_with_llm(documents, 100, 200, 4000)

In [34]:
for chunk in chunks:
    print(chunk.page_content)
    print("-----")

 The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3,2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market. Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-handling skills for physicians hesitant to transition from established tr

### J. Search the new chunks

Changing `chunks` does not update the database. The next cells delete the old
collection and index the new chunks. We then retrieve one chunk and rebuild
the same RAG chain.

One new chunk can hold much more text than an earlier chunk. We have changed
both the chunk boundaries and `k`, so this comparison does not isolate either
change on its own.

In [35]:
# clear the vectorstore
vectorstore.delete_collection()

In [36]:
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings(model="openai.text-embedding-3-large"))

In [37]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [39]:
rag_chain.invoke("Is Zelomax safe during pregnancy?")

'The safety of Zelomax during pregnancy is uncertain due to limited long-term data. Clinicians recommend discussing its safety profile with your healthcare provider to weigh potential benefits and risks. It is best to consult your care team for personalized advice.'

### Your observations 3

Find a new chunk that keeps related information together and compare it with
the earlier split. Did that change help the answer in your run? Explain whether
the benefit you observed seems worth the additional model calls. Remember that
this example also changed `k`, so chunking alone cannot explain every difference.

**Your response:**

_better chunking helped change the answer, new chunks keep related information together and it was worth using LLM to create the chunks_

## Part 2: RAG with a structured database

For a question such as “How many employees are there?”, retrieving a few similar
records would not give us the total. SQL can count the records directly.
We will let a model write SQL, execute it, and include the results in an
answering prompt.

### K. Open and inspect the database

[Chinook](https://github.com/lerocha/chinook-database) is a sample music-store
database with artists, albums, tracks, customers, employees, and sales.
`SQLDatabase` lets LangChain inspect its tables and run queries. SQLite reads
from the supplied file; `mode=ro` opens it read-only.

The first query selects up to ten rows from `Artist`. pandas displays the
result as a table. No model call is needed for these steps.

In [40]:
# These helpers display formatted output in the notebook.
# We use Markdown below to render the model's answers.
from IPython.display import display, Markdown, HTML


In [41]:
# SQLDatabase lets LangChain inspect tables and run SQL queries.
# pandas displays query results as a table; Path helps locate our database file.
from langchain_community.utilities import SQLDatabase
from IPython.display import display
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# SQLite stores this entire sample database in one file, so no server is needed.
# The first path works from this notebook's folder; the second works from the repo root.
db_path = Path("data/Chinook.db")
if not db_path.is_file():
    db_path = Path("assignments/02-structured-data-rag/data/Chinook.db")
if not db_path.is_file():
    raise FileNotFoundError("Open this notebook from the repository root or assignments/02-structured-data-rag folder.")

# This connection address identifies SQLite and the full path to the file.
# mode=ro opens it read-only, so generated queries cannot change its records.
db_uri = f"sqlite:///file:{db_path.resolve().as_posix()}?mode=ro&uri=true"
db = SQLDatabase.from_uri(db_uri)

# The dialect tells us which variety of SQL to use. The table names tell us
# what data we can query. None of these operations calls a language model.
display(db.dialect)
display(db.get_usable_table_names())

# An engine manages database connections. pandas uses this one to execute SQL.
engine = create_engine(db_uri)

# Start with a query we wrote ourselves: show up to 10 rows from Artist.
query = "SELECT * FROM Artist LIMIT 10;"
df = pd.read_sql_query(query, engine)
display(df)


'sqlite'

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains
5,6,Antônio Carlos Jobim
6,7,Apocalyptica
7,8,Audioslave
8,9,BackBeat
9,10,Billy Cobham


### L. Generate SQL and inspect the prompt

We now create the model interface for the database example. This reassigns
`llm`; the next cells also reuse `chain` and `response` as the example develops.
Those names are ordinary Python variables.

`create_sql_query_chain` sends the question, table definitions, and sample rows
to the model. It returns SQL text; it does not execute the SQL. The last cell
shows the prompt template that LangChain uses to request it.

In [42]:
# ChatOpenAI is LangChain's interface to the model.
# It reads OPENAI_API_KEY and OPENAI_BASE_URL from the Codespace environment.
from langchain_openai import ChatOpenAI
from os import environ

# This creates the model interface. A later .invoke(...) call sends the request.
llm = ChatOpenAI(model="openai.gpt-4o")


In [43]:
from langchain.chains import create_sql_query_chain

# A chain connects processing steps. This one sends the question, table
# definitions, and sample rows to the model so it can write SQLite SQL.
# It returns SQL text; it does not execute the generated query.
chain = create_sql_query_chain(llm, db)
response = chain.invoke({"question": "What are the three biggest hits?"})

# response is just our variable name. Here it contains the generated SQL.
# What did the model decide "biggest hits" means: sales, revenue, or something else?
display(Markdown(response))


```sql
SQLQuery: 
SELECT "Name", "Milliseconds"
FROM "Track"
ORDER BY "Milliseconds" DESC
LIMIT 3;
```

Assuming the query is executed, and the result is:

```plaintext

In [44]:
# Inspect the prompt template chosen by LangChain for this SQL database.
# Look for the SQL instructions and the input/table_info placeholders.
# The chain fills those placeholders when invoked. This cell makes no model call.
chain.get_prompts()[0].pretty_print()


You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

### Your observations 4

What did the model take “the three biggest hits” to mean? Point to a column,
calculation, or ordering rule in its SQL. Then write a more precise version of
the question and identify where the prompt supplies the table information the
model needs. You only need to write your revised question here.

**Your response:**

_The model takes the biggest hits and puts them into the most purchased albums, so it's calculated like the highest total sales of the tracks. a revised question can be What are the three tracks with the highest total number of units sold across all invoices?_

### M. Clean and execute the SQL

A model may wrap SQL in Markdown or add a label. `clean_sql_query` removes
common formatting so SQLite can read the query. It does not check whether
the query answers our question correctly.

Inspect `clean_response` before running it. `db.run` executes the SQL and
returns the result rows as text.

In [45]:
# re provides text-pattern matching. This helper handles response formatting;
# it does not verify SQL correctness or decide whether the query answers the question.
import re

def clean_sql_query(response):
    # Remove surrounding whitespace, then extract SQL from a code block if present.
    query = response.strip()
    fenced_sql = re.search(r"```(?:sql)?\s*(.*?)```", query, re.DOTALL | re.IGNORECASE)
    if fenced_sql:
        query = fenced_sql.group(1).strip()

    # The chain can stop before a closing code fence is generated.
    # Also remove a leading SQLQuery: label, which SQLite would not understand.
    query = re.sub(r"^```(?:sql)?\s*", "", query, flags=re.IGNORECASE)
    query = re.sub(r"^SQLQuery:\s*", "", query, flags=re.IGNORECASE)

    # Skip introductory prose and find a SELECT query or a WITH clause.
    # WITH names a temporary query result that the rest of the SQL can reference.
    sql_start = re.search(r"^\s*(?:SELECT|WITH)\b", query, re.MULTILINE | re.IGNORECASE)
    if sql_start is None:
        raise ValueError(f"No SELECT or WITH query found in model output: {response!r}")
    return query[sql_start.start():].strip()


In [46]:
# Clean the SQL we generated earlier and inspect it before running it.
# This is ordinary Python text processing, with no additional model call.
clean_response = clean_sql_query(response)
print(clean_response)


SELECT "Name", "Milliseconds"
FROM "Track"
ORDER BY "Milliseconds" DESC
LIMIT 3;


In [47]:
# Now execute the query against SQLite. The database supplies the actual records.
# In this example, db.run returns a string showing the result rows as tuples.
# Check the query's joins, ordering, and limits before interpreting those rows.
db.run(clean_response)


"[('Occupation / Precipice', 5286953), ('Through a Looking Glass', 5088838), ('Greetings from Earth, Pt. 1', 2960293)]"

### N. Connect SQL generation and execution

`QuerySQLDataBaseTool` lets us include database execution in a chain.
The sequence is **write SQL → clean SQL → execute SQL**. Its output is still
the database result, not a model-written explanation. For example, `[(8,)]`
is a result containing one row with the value 8.

The tool can return an SQL error as text, so read the output even when the
Python cell finishes without an exception.

In [48]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

# Wrap SQL execution as a LangChain tool so we can connect it to other steps.
# A tool here is called by our pipeline; the model is not choosing when to use it.
execute_query = QuerySQLDataBaseTool(db=db)
write_query = create_sql_query_chain(llm, db)

# The | operator passes each step's output to the next step:
# question dictionary -> SQL text -> cleaned SQL -> database result.
# We now assign chain to this larger pipeline, replacing the earlier SQL-only chain.
chain = write_query | clean_sql_query | execute_query

# This returns the query result, before any model-written explanation.
# The tool returns SQL errors as text too, so inspect the output for errors.
chain.invoke({"question": "How many employees are there"})


'[(8,)]'

### O. Turn the result into an answer

The answering prompt receives the original question, SQL, and database result.
`RunnablePassthrough.assign` keeps the existing dictionary fields while adding
`query` and then `result`. `itemgetter("query")` selects the SQL for execution.
Finally, the prompt and model turn the result into an answer.

In [49]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# The answering model needs the original question, the SQL, and its result.
# The names inside braces are placeholders filled from a dictionary at runtime.
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

# Keep the original question as we add information to that dictionary:
# {question} -> {question, query} -> {question, query, result}.
# assign adds a field; itemgetter("query") selects just the SQL for execution.
chain = (
    RunnablePassthrough.assign(query=write_query | clean_sql_query)
    .assign(
        result=itemgetter("query") | execute_query
    )
    | answer_prompt      # Fill the template with the question, SQL, and result.
    | llm                # Ask the model to explain the database result.
    | StrOutputParser()  # Extract the model message's text as a Python string.
)

# Building this pipeline does not run it. Each invocation below makes two
# model calls: one to write SQL and one to turn the result into an answer.


In [50]:
# Run the full pipeline and compare its answer with the raw employee count above.
# response now holds the final answer, because chain now includes the answering step.
response = chain.invoke({"question": "How many employees are there?"})

display(Markdown(response))


There are 8 employees.

In [51]:
# The pipeline writes a fresh query each time; it does not reuse the earlier SQL.
# There is no conversation history here. This question is answered independently.
response = chain.invoke({"question": "What are the three biggest hits?"})

# Did the model use the same meaning of "biggest hits" as before?
# Try specifying "tracks ranked by total units sold across all invoices" instead.
display(Markdown(response))


The three biggest hits are:  
1. "Occupation / Precipice" with a duration of 5,286,953 milliseconds.  
2. "Through a Looking Glass" with a duration of 5,088,838 milliseconds.  
3. "Greetings from Earth, Pt. 1" with a duration of 2,960,293 milliseconds.

### Your observations 5

How many model calls does one invocation of this full chain make? Trace what
each call receives and produces, and identify the step performed by SQLite.
Use the employee-count example to explain why generating SQL is different from
retrieving the answer.

**Your response:**

_One invocation of the full chain makes two model calls. The first model call receives the user’s question and the database schema, and produces a SQL query. SQLite then executes that SQL query and returns the actual database result. The second model call receives the original question, the generated SQL, and the SQL result, and turns them into a natural-language answer. For the employee-count example, the model only generates a query such as `SELECT COUNT(*) FROM Employee`, while SQLite is what actually retrieves the number of employees from the database._

### P. Consider the limits of the evidence

The final examples ask for a promotion recommendation. Sales records do not
establish an employee's awareness of trends or whether they deserve a promotion.
Also consider what “last year” means for a database of historical records.

The model might generate SQL, decline, or produce text that the cleanup helper
cannot turn into SQL. If the full chain raises that error, record it and run
the following SQL-only cell separately. That cell makes a new request; it does
not reveal the exact query used in the earlier call.

In [52]:
# This question goes beyond what sales records can establish.
# A high sales count does not prove awareness of trends or merit a promotion.
# Also ask what date range "last year" should mean for this historical dataset.
response = chain.invoke({"question": "Which genre generated the most total revenue in each country?"})

# Check whether the answer separates observed sales from unsupported conclusions.
display(Markdown(response))


Based on the SQL result, the genre that generated the most total revenue in **Argentina** is a **tie** between **Rock** and **Alternative & Punk**, each generating **$8.91** in total revenue.

If you are looking for results from other countries, they are not included in the provided SQL result, which only shows data for Argentina.

In [53]:
# Return to a SQL-only chain to examine how a similar question becomes a query.
# This generates new SQL; it does not reveal the exact SQL used in the previous cell.
test_chain = create_sql_query_chain(llm, db)
response = test_chain.invoke({"question": "Which of our sales support agents should be promoted based on their awareness of trending genres, can be classified based on the last year of sales??"})

# Inspect the date filter, joins, and ranking rule. What can those actually measure?
# No SQL is executed by this cell.
display(Markdown(response))


To determine which sales support agent should be promoted based on their awareness of trending genres, I'll analyze the sales (from the `Invoice` table) made in the last year and group them by the sales support agents (from the `Employee` table via `Customer.SupportRepId`). I will then classify the agents based on the total sales in the last year.

Here is the query to retrieve the data:

```sql
SQLQuery: 
SELECT 
    "E"."FirstName" || ' ' || "E"."LastName" AS "EmployeeName",
    SUM("I"."Total") AS "TotalSales"
FROM 
    "Employee" AS E
JOIN 
    "Customer" AS C ON "E"."EmployeeId" = "C"."SupportRepId"
JOIN 
    "Invoice" AS I ON "C"."CustomerId" = "I"."CustomerId"
WHERE 
    "I"."InvoiceDate" >= date('now', '-1 year')
GROUP BY 
    "E"."EmployeeId"
ORDER BY 
    "TotalSales" DESC
LIMIT 5;
```

Now, let's analyze the results:

```plaintext

### Your observations 6

Try the challenge from class: ask a question that the database can answer but
that might lead the model to an incorrect answer. Choose a question involving
a comparison, a grouping, or more than one table; a question about someone's
private preferences would not test this.

Replace the question in an existing SQL-only cell to inspect the generated query,
then try it in an existing full-chain cell. These are separate requests and may
generate different SQL. In a few sentences, record your question, the answer,
and the evidence you used to check it. If you did not find a mistake, explain
what you checked rather than assuming a fluent answer is correct.

**Your response:**

_question: Which genre generated the most total revenue in each country? answer: Based on the SQL result, the genre that generated the most total revenue in Argentina is a tie between Rock and Alternative & Punk, each generating $8.91 in total revenue.

If you are looking for results from other countries, they are not included in the provided SQL result, which only shows data for Argentina. 

It may have grouped the data correctly but then applied a global limit, causing only Argentina or a few rows to appear._

## Save your work

Save this notebook with your outputs and six responses. Keep the supplied data
files unchanged, then commit and push the notebook to your fork. Follow Canvas
for the submission link and deadline.